In [1]:
%pip install mediapipe 
%pip install opencv-python 
%pip install numpy 
%pip install tqdm
%pip install scipy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Imports

In [2]:
import cv2
import json
import numpy as np
from tqdm import tqdm
from scipy.spatial.transform import Rotation as R

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

Configuração

In [3]:
VIDEO_PATH = "Video/video.mp4"
MODEL_PATH = "Models/pose_landmarker_lite.task"

JSON_OUTPUT = "Json/pose_landmarks_v3.json"
BVH_OUTPUT = "bvh/animation.bvh"
PREVIEW_OUTPUT = "Video/preview_motion.mp4"

Verificações

In [4]:
BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
RunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_poses=1
)

Extração dos landmarks

In [5]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(
    PREVIEW_OUTPUT,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

SMOOTHING (MOCAP)

In [6]:
alpha = 0.75
smoothed = {}

def smooth(i, p):
    if i not in smoothed:
        smoothed[i] = p
        return p

    smoothed[i] = alpha * smoothed[i] + (1 - alpha) * p
    return smoothed[i]

ESQUELETO - Blender/Unity rig view

In [7]:
POSE_CONNECTIONS = [
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),

    (11, 23), (12, 24),
    (23, 24),

    (23, 25), (25, 27),
    (24, 26), (26, 28),
]

'''
POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),

    (9, 10),

    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),

    (11, 23), (12, 24),
    (23, 24),

    (23, 25), (25, 27),
    (24, 26), (26, 28),

    (27, 29), (29, 31),
    (28, 30), (30, 32),
]
'''

'\nPOSE_CONNECTIONS = [\n    (0, 1), (1, 2), (2, 3), (3, 7),\n    (0, 4), (4, 5), (5, 6), (6, 8),\n\n    (9, 10),\n\n    (11, 12),\n    (11, 13), (13, 15),\n    (12, 14), (14, 16),\n\n    (11, 23), (12, 24),\n    (23, 24),\n\n    (23, 25), (25, 27),\n    (24, 26), (26, 28),\n\n    (27, 29), (29, 31),\n    (28, 30), (30, 32),\n]\n'

helpers

In [8]:
def npv(frame, idx):
    return np.array(frame[idx])

def midpoint(a, b):
    return (a + b) / 2.0

ROTATION CORRIGIDA (IMPORTANTE PARA BVH)

In [9]:
def bone_rotation(parent, child):
    v = child - parent
    n = np.linalg.norm(v)

    if n < 1e-8:
        return np.zeros(3)

    v = v / n

    ref = np.array([0, 1, 0])

    axis = np.cross(ref, v)
    axis_norm = np.linalg.norm(axis)

    if axis_norm < 1e-8:
        return np.zeros(3)

    axis = axis / axis_norm

    angle = np.arccos(np.clip(np.dot(ref, v), -1, 1))

    rot = R.from_rotvec(axis * angle)

    return rot.as_euler("ZXY", degrees=True)

BVH HIERARCHY (BLENDER SAFE)

✔ compatível com humanoide
✔ import direto no Blender

In [ ]:
print("EXPECTED:", 6 + 3*4)
print("ACTUAL:", len(motion_lines[0].split()))

Cabeçalho BVH

In [ ]:
BVH_HEADER = """
HIERARCHY
ROOT Hips
{
    OFFSET 0 0 0
    CHANNELS 6 Xposition Yposition Zposition Zrotation Xrotation Yrotation

    JOINT LeftUpLeg
    {
        OFFSET -5 -10 0
        CHANNELS 3 Zrotation Xrotation Yrotation

        JOINT LeftLeg
        {
            OFFSET 0 -15 0
            CHANNELS 3 Zrotation Xrotation Yrotation

            End Site { OFFSET 0 -15 0 }
        }
    }

    JOINT RightUpLeg
    {
        OFFSET 5 -10 0
        CHANNELS 3 Zrotation Xrotation Yrotation

        JOINT RightLeg
        {
            OFFSET 0 -15 0
            CHANNELS 3 Zrotation Xrotation Yrotation

            End Site { OFFSET 0 -15 0 }
        }
    }
}
"""

STORAGE

In [11]:
frames = []

Processamento

In [12]:
with PoseLandmarker.create_from_options(options) as landmarker:

    for i in tqdm(range(frame_count)):

        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        ts = int(i * 1000 / fps)

        result = landmarker.detect_for_video(mp_image, ts)

        pose_data = []
        points = []

        if result.pose_landmarks:

            pose = result.pose_landmarks[0]

            for idx, lm in enumerate(pose):

                x = lm.x * width
                y = lm.y * height

                p = smooth(idx, np.array([x, y]))

                points.append((int(p[0]), int(p[1])))

                pose_data.append([lm.x, lm.y, lm.z])

                cv2.circle(frame, (int(p[0]), int(p[1])), 3, (0,255,0), -1)

            for a, b in POSE_CONNECTIONS:
                if a < len(points) and b < len(points):
                    cv2.line(frame, points[a], points[b], (255,180,0), 2)

        frames.append(pose_data)
        writer.write(frame)

100%|█████████▉| 246/247 [00:09<00:00, 25.07it/s]


JSON

In [13]:
cap.release()
writer.release()

with open(JSON_OUTPUT, "w") as f:
    json.dump(frames, f)

BVH EXPORT

In [ ]:
motion_lines = []

for frame in frames:

    if len(frame) < 33:
        continue

    lhip = npv(frame, 23)
    rhip = npv(frame, 24)

    lknee = npv(frame, 25)
    rknee = npv(frame, 26)

    lankle = npv(frame, 27)
    rankle = npv(frame, 28)

    mid_hip = midpoint(lhip, rhip)
    root = mid_hip * 100

    left_upper = bone_rotation(lhip, lknee)
    left_lower = bone_rotation(lknee, lankle)

    right_upper = bone_rotation(rhip, rknee)
    right_lower = bone_rotation(rknee, rankle)

    values = [
        root[0], root[1], root[2],  # ROOT 6 channels total

        0, 0, 0,  # ROOT rotation

        *left_upper,   # LeftUpLeg (3)
        *left_lower,   # LeftLeg (3)

        *right_upper,  # RightUpLeg (3)
        *right_lower   # RightLeg (3)
    ]

    motion_lines.append(" ".join(f"{v:.6f}" for v in values))

WRITE BVH

In [15]:
with open(BVH_OUTPUT, "w") as f:
    f.write(BVH_HEADER)
    f.write("\nMOTION\n")
    f.write(f"Frames: {len(motion_lines)}\n")
    f.write(f"Frame Time: {1/fps:.6f}\n")

    for line in motion_lines:
        f.write(line + "\n")